# Hugging Face DataSet + Chunks

In diesem Notebook werden die Markdown-Dateien nicht mehr als ganze Dokumente verwendet, sondern in kleinere Textabschnitte (Chunks) aufgeteilt. Chunks teilen längere Texte in kleinere Abschnitte auf, sodass ein System gezielter die Textstelle finden kann, die eine Frage tatsächlich beantwortet.

Auf Basis dieser Chunks werden anschliessend Embeddings erzeugt. Embeddings wandeln Text in eine Zahlenrepräsentation um, sodass ein Computer die Bedeutung von Wörtern und Sätzen vergleichen kann. Dadurch lassen sich Fragen mit inhaltlich passenden Textstellen finden, auch wenn nicht exakt die gleichen Wörter verwendet werden.

In [ ]:
from datasets import load_from_disk

DATASET_DIR = "dataset"

ds = load_from_disk(DATASET_DIR)

print(ds)

In [ ]:
def chunk_text(text, chunk_size=800, overlap=200):
    chunks = []
    start = 0
    length = len(text)

    while start < length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [ ]:
chunk_rows = []

for row in ds:
    chunks = chunk_text(row["content"])
    
    for i, c in enumerate(chunks):
        chunk_rows.append({
            "path": row["path"],
            "chunk_id": i,
            "chunk": c
        })

print("Chunks:", len(chunk_rows))

In [ ]:
from datasets import Dataset
from pathlib import Path
import shutil

chunk_ds = Dataset.from_list(chunk_rows)

print(chunk_ds)
print(chunk_ds[0])

OUTPUT_DIR = Path("dataset_chunks")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

chunk_ds.save_to_disk(str(OUTPUT_DIR))

print(f"Chunk-Dataset gespeichert unter: {OUTPUT_DIR}")
print(f"Anzahl Chunks: {len(chunk_ds)}")


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

texts = chunk_ds["chunk"]

embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)

print("Embedding shape:", embeddings.shape)

In [ ]:
def search(question, top_k=3):
    q_emb = model.encode([question], convert_to_numpy=True)[0]

    doc_norms = np.linalg.norm(embeddings, axis=1)
    q_norm = np.linalg.norm(q_emb)

    scores = (embeddings @ q_emb) / (doc_norms * q_norm + 1e-12)

    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_idx:
        results.append({
            "path": chunk_ds[idx]["path"],
            "chunk_id": chunk_ds[idx]["chunk_id"],
            "chunk": chunk_ds[idx]["chunk"],
            "score": float(scores[idx])
        })

    return results

In [ ]:
def answer(question, top_k=3):
    results = search(question, top_k)

    print("Frage:", question)
    print()

    for i, r in enumerate(results, 1):
        print(f"Treffer {i} | {r['path']} | Chunk {r['chunk_id']} | Score {r['score']:.3f}")
        print(r["chunk"])
        print("\n" + "-" * 70 + "\n")

In [ ]:
answer("LernMAAS: Mein Modul/Kurs ist noch nicht vorhanden, wie komme ich zu meinem Modul")

In [ ]:
answer("Projekt LernCloud macht?")